# Fine-Tuning BERT for Question Answering on SQuAD v1.1


In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
import torch, numpy as np, collections
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Implementation

### Step 1 - Environment setup


In [1]:
USE_SUBSET = False      # True = fast test run (small data), False = full training
N_TRAIN = 5000          # used only when USE_SUBSET = True
N_VAL = 1000

### Step 2 - Load and explore the dataset
SQuAD v1.1 contains Wikipedia paragraphs, with questions written by crowd-workers. Every answer is a span of the paragraph.
It has ~87.6k training and ~10.6k validation examples.

In [3]:
from datasets import load_dataset
dataset = load_dataset("rajpurkar/squad")
print(dataset)

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [4]:
# Look at one training example
example = dataset["train"][0]
for k, v in example.items():
    print(f"{k}: {v}\n")

id: 5733be284776f41900661182

title: University_of_Notre_Dame

context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

answers: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}



Each example has:
* `question`: the question text
* `context`: the paragraph containing the answer
* `answers`: a dict with `text` (the answer string) and `answer_start` (the **character** index in the context where the answer begins)

Let's verify that the character position really points to the answer:

In [5]:
ctx = example["context"]
start = example["answers"]["answer_start"][0]
text = example["answers"]["text"][0]
print("Answer from context slice:", ctx[start:start+len(text)])
print("Answer text field        :", text)

Answer from context slice: Saint Bernadette Soubirous
Answer text field        : Saint Bernadette Soubirous


In [6]:
# Basic statistics
import pandas as pd
df = pd.DataFrame(dataset["train"])
print("Train size:", len(dataset["train"]), "| Validation size:", len(dataset["validation"]))
print("Avg context length (words):", df["context"].str.split().str.len().mean().round(1))
print("Avg question length (words):", df["question"].str.split().str.len().mean().round(1))

Train size: 87599 | Validation size: 10570
Avg context length (words): 119.8
Avg question length (words): 10.1


In [7]:
if USE_SUBSET:
    dataset["train"] = dataset["train"].shuffle(seed=42).select(range(N_TRAIN))
    dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(N_VAL))
print({k: len(v) for k, v in dataset.items()})

{'train': 87599, 'validation': 10570}


### Step 3 - Load the tokenizer and model


In [8]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

### Step 4 - Tokenization and preprocessing (the most important step)

BERT takes the input as `[CLS] question [SEP] context [SEP]` and has a **maximum length of 512 tokens** (we use 384 to save memory).
Two problems must be solved:

**Problem 1 - Long contexts.** Some contexts are longer than 384 tokens. If we just cut them, the answer might be lost.
Solution: split the context into **overlapping windows** (`stride=128`, `return_overflowing_tokens=True`). One example can therefore produce several training features.

**Problem 2 - Labels are in characters, the model needs tokens.** SQuAD gives `answer_start` as a character index, but the model predicts token indexes.
Solution: use `return_offsets_mapping=True`, which tells us, for each token, the `(start_char, end_char)` it covers in the original text. We then find
which tokens cover the answer's start and end characters.

If the answer is **not inside a given window**, we label that window with `(0, 0)`, which points to the `[CLS]` token, meaning "no answer here".

In [9]:
max_length = 384   # max tokens per feature
doc_stride = 128   # overlap between windows of a long context

def prepare_train_features(examples):
    # Remove leading whitespace in questions (it can confuse the tokenizer)
    questions = [q.lstrip() for q in examples["question"]]

    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",        # only cut the context, never the question
        stride=doc_stride,
        return_overflowing_tokens=True,  # long context -> several features
        return_offsets_mapping=True,     # token -> character positions
        padding="max_length",
    )

    # Which original example each feature came from
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # sequence_ids: 0 = question tokens, 1 = context tokens, None = special tokens
        sequence_ids = tokenized.sequence_ids(i)

        answers = examples["answers"][sample_map[i]]
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        # Find where the context begins and ends inside this feature
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # Is the answer fully inside this window?
        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)   # answer not here -> point to [CLS]
            end_positions.append(cls_index)
        else:
            # move start pointer to the token containing the first answer character
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)
            # move end pointer to the token containing the last answer character
            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

In [10]:
train_dataset = dataset["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
print(train_dataset)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 88524
})


**Sanity check:** decode the tokens between the labelled start and end positions. They should equal the real answer.

In [11]:
f = train_dataset[0]
decoded = tokenizer.decode(f["input_ids"][f["start_positions"]: f["end_positions"] + 1])
print("Decoded answer from labels:", decoded)
print("True answer               :", dataset["train"][0]["answers"]["text"][0])

Decoded answer from labels: saint bernadette soubirous
True answer               : Saint Bernadette Soubirous


In [12]:
def prepare_validation_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["example_id"].append(examples["id"][sample_map[i]])
        # keep offsets only for context tokens
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]
    return tokenized

validation_dataset = dataset["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=dataset["validation"].column_names,
)
print(validation_dataset)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 10784
})


### Step 5 - Fine-tuning with the Hugging Face `Trainer`


In [17]:
import torch, numpy as np, collections
from transformers import TrainingArguments, Trainer, default_data_collator

epochs, bs = 2, 16
total_steps = (len(train_dataset) // bs) * epochs
warmup_steps = int(0.1 * total_steps)   # 10% warmup

args = TrainingArguments(
    output_dir="bert-finetuned-squad",
    num_train_epochs=epochs,
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    logging_steps=200,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=default_data_collator,
    processing_class=tokenizer,
)

In [18]:
train_result = trainer.train()
print(train_result.metrics)

Step,Training Loss
200,5.359564
400,3.378768
600,2.277097
800,1.847082
1000,1.631754
1200,1.520983
1400,1.455284
1600,1.378769
1800,1.374441
2000,1.294782


{'train_runtime': 3929.8, 'train_samples_per_second': 45.053, 'train_steps_per_second': 2.816, 'total_flos': 3.4696551139946496e+16, 'train_loss': 1.1258673634297287, 'epoch': 2.0}


### Step 6 - Evaluation

In [19]:
# 1) Predict logits
raw_predictions = trainer.predict(validation_dataset.remove_columns(["example_id", "offset_mapping"]))
start_logits, end_logits = raw_predictions.predictions
print(start_logits.shape, end_logits.shape)

(10784, 384) (10784, 384)


In [21]:
# 2) Post-processing: logits -> answer text
n_best = 20            # consider the top-20 start and end tokens
max_answer_length = 30 # answers longer than 30 tokens are discarded

def postprocess_predictions(examples, features, start_logits, end_logits):
    # group features by the example they belong to
    features_per_example = collections.defaultdict(list)
    for i, feat in enumerate(features):
        features_per_example[feat["example_id"]].append(i)

    predicted = {}
    for example in examples:
        context = example["context"]
        best = {"score": -1e9, "text": ""}

        for fi in features_per_example[example["id"]]:
            s_logits = start_logits[fi]
            e_logits = end_logits[fi]
            offsets = features[fi]["offset_mapping"]

            start_idxs = np.argsort(s_logits)[-1: -n_best - 1: -1].tolist()
            end_idxs = np.argsort(e_logits)[-1: -n_best - 1: -1].tolist()

            for s in start_idxs:
                for e in end_idxs:
                    # skip positions outside the context
                    if offsets[s] is None or offsets[e] is None:
                        continue
                    # skip invalid spans
                    if e < s or e - s + 1 > max_answer_length:
                        continue
                    score = s_logits[s] + e_logits[e]
                    if score > best["score"]:
                        best = {"score": score,
                                "text": context[offsets[s][0]: offsets[e][1]]}

        predicted[example["id"]] = best["text"]
    return predicted

predicted_answers = postprocess_predictions(
    dataset["validation"], validation_dataset, start_logits, end_logits
)
print("Predictions made:", len(predicted_answers))

Predictions made: 10570


In [24]:
!pip install -q evaluate

In [25]:
# 3) Score with the official SQuAD metric
import evaluate
squad_metric = evaluate.load("squad")

predictions = [{"id": k, "prediction_text": v} for k, v in predicted_answers.items()]
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in dataset["validation"]]

results = squad_metric.compute(predictions=predictions, references=references)
print("Validation Exact Match :", round(results["exact_match"], 2))
print("Validation F1 Score    :", round(results["f1"], 2))

Validation Exact Match : 81.01
Validation F1 Score    : 88.32


## 3. Results

The cell below prints a clean results table. For reference, a well-trained `bert-base-uncased` on the full SQuAD v1.1
usually reaches about **EM = 80-81** and **F1 = 88**.

In [26]:
results_df = pd.DataFrame({
    "Metric": ["Exact Match (EM)", "F1 Score"],
    "Validation Score": [round(results["exact_match"], 2), round(results["f1"], 2)],
})
results_df

,Metric,Validation Score
0,Exact Match (EM),81.01
1,F1 Score,88.32


### Custom question-answer tests
We test the fine-tuned model on our own questions. Manual method: tokenize, run the model, take the `argmax` of the start and end logits,
and decode the tokens between them. (This is exactly the idea in the assignment's example; the extra step is converting logits into an answer.)

In [27]:
def answer_question(question, context):
    model.eval()
    inputs = tokenizer(question, context, return_tensors="pt", truncation="only_second", max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    start_idx = int(torch.argmax(outputs.start_logits))
    end_idx = int(torch.argmax(outputs.end_logits))
    if end_idx < start_idx:
        return "(no valid answer found)"
    return tokenizer.decode(inputs["input_ids"][0][start_idx:end_idx + 1], skip_special_tokens=True)

tests = [
    ("Who developed the theory of relativity?",
     "Albert Einstein developed the theory of relativity in the early 20th century."),
    ("Where is the Eiffel Tower located?",
     "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was completed in 1889."),
    ("When was the Eiffel Tower completed?",
     "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It was completed in 1889."),
    ("What is the capital of Bangladesh?",
     "Dhaka is the capital and largest city of Bangladesh. It is located on the banks of the Buriganga River."),
    ("Which river is Dhaka located on?",
     "Dhaka is the capital and largest city of Bangladesh. It is located on the banks of the Buriganga River."),
]

for q, c in tests:
    print("Q:", q)
    print("A:", answer_question(q, c))
    print("-" * 60)

Q: Who developed the theory of relativity?
A: albert einstein
------------------------------------------------------------
Q: Where is the Eiffel Tower located?
A: paris, france
------------------------------------------------------------
Q: When was the Eiffel Tower completed?
A: 1889
------------------------------------------------------------
Q: What is the capital of Bangladesh?
A: dhaka
------------------------------------------------------------
Q: Which river is Dhaka located on?
A: buriganga river
------------------------------------------------------------


The same test using the high-level Hugging Face `pipeline`, which does tokenization, long-context handling, and post-processing automatically
and also returns a confidence score:

In [29]:
def answer_with_score(question, context, n_best=20, max_answer_len=30):
    model.eval()
    inputs = tokenizer(question, context, return_tensors="pt",
                       truncation="only_second", max_length=512,
                       return_offsets_mapping=True)
    offsets = inputs.pop("offset_mapping")[0].tolist()
    seq_ids = inputs.sequence_ids(0)          # 1 = context tokens
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs)

    start_probs = torch.softmax(out.start_logits[0], dim=-1).cpu().numpy()
    end_probs = torch.softmax(out.end_logits[0], dim=-1).cpu().numpy()

    best = {"answer": "(no answer found)", "score": 0.0}
    for s in np.argsort(start_probs)[-n_best:]:
        for e in np.argsort(end_probs)[-n_best:]:
            if seq_ids[s] != 1 or seq_ids[e] != 1:   # must be inside the context
                continue
            if e < s or e - s + 1 > max_answer_len:
                continue
            score = float(start_probs[s] * end_probs[e])
            if score > best["score"]:
                best = {"answer": context[offsets[s][0]:offsets[e][1]], "score": round(score, 4)}
    return best

print(answer_with_score(
    "Who developed the theory of relativity?",
    "Albert Einstein developed the theory of relativity in the early 20th century."))

for q, c in tests:
    print("Q:", q, "->", answer_with_score(q, c))

{'answer': 'Albert Einstein', 'score': 0.9969}
Q: Who developed the theory of relativity? -> {'answer': 'Albert Einstein', 'score': 0.9969}
Q: Where is the Eiffel Tower located? -> {'answer': 'Paris, France', 'score': 0.754}
Q: When was the Eiffel Tower completed? -> {'answer': '1889', 'score': 0.979}
Q: What is the capital of Bangladesh? -> {'answer': 'Dhaka', 'score': 0.9713}
Q: Which river is Dhaka located on? -> {'answer': 'Buriganga River', 'score': 0.7271}


In [31]:
# Optional: save the model so you can reload it later
trainer.save_model("bert-finetuned-squad-final")
tokenizer.save_pretrained("bert-finetuned-squad-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert-finetuned-squad-final/tokenizer_config.json',
 'bert-finetuned-squad-final/tokenizer.json')

## 5. Conclusion

In this assignment, I fine-tuned `bert-base-uncased` on SQuAD v1.1 for extractive question answering using the Hugging Face `datasets`, `Trainer` and `evaluate` tools. Unlike classification, where a model predicts one label, a QA model predicts the start and end tokens of the answer inside the context. The main technical step was preprocessing: converting character-based answer positions into token positions with offset mappings, and handling long contexts with a sliding window (`max_length=384`, `stride=128`).

After 2 epochs (batch size 16, learning rate 3e-5), the model reached an **Exact Match of 81.01** and an **F1 score of 88.32** on the validation set, which matches typical published results for BERT-base. The gap between EM and F1 shows that the model usually finds the right part of the text but sometimes differs slightly in the answer's boundaries. All five of my custom tests were answered correctly, for example "Albert Einstein" (confidence 0.997) and "1889" (confidence 0.979), and lower confidence appeared where answer boundaries were less clear, such as "Paris, France" (0.754).

Overall, this project showed that a pre-trained transformer can be adapted to a new task with a small task-specific layer and a short period of fine-tuning. One limitation is that SQuAD v1.1 always contains an answer, so the model cannot say "no answer" when the context lacks one. Future work could use SQuAD v2.0, try larger models such as RoBERTa or DeBERTa, or test on text from other domains.